# Guess the Number Game
The game includes:
- Three difficulty levels with different numbers of guesses.
- Helpful feedback a valid guess.
- A hint after three incorrect guesses.
- High-score saving in a text file.
- A leaderboard showing the best scores.


In [17]:
# Import Path so the score file works across operating systems.
from pathlib import Path

# Store the allowed difficulty names and their maximum valid guesses.
DIFFICULTY_LIMITS = {
    "easy": 10,
    "medium": 7,
    "hard": 5,
}

# Keep scores in a text file beside the notebook.
SCORES_FILE = Path("guess_number_scores.txt")

# Confirm that the shared game settings have been created.
print("Game setup is ready.")


Game setup is ready.


## 1. Choose a difficulty level

The player chooses **Easy**, **Medium**, or **Hard**. The input is normalized to lowercase so entries such as `EASY` also work.

The selected level determines the maximum number of valid guesses:

- Easy: 10 guesses
- Medium: 7 guesses
- Hard: 5 guesses


In [ ]:
def choose_difficulty():
    """Ask until the player enters one of the available difficulty levels."""
    # Keep asking until the input matches one of the configured difficulty names.
    while True:
        # Remove extra spaces and ignore capitalization differences.
        difficulty = input("Difficulty (Easy/Medium/Hard): ").strip().lower()
        if difficulty in DIFFICULTY_LIMITS:
            # Return the normalized name so the rest of the game can use it.
            return difficulty
        # Explain the accepted choices when the player enters invalid input.
        print("Please choose Easy, Medium, or Hard.")


# Uncomment these lines to test difficulty selection by itself.
# difficulty = choose_difficulty()
# print(f"You selected {difficulty.title()}.")


## 2. Generate a number and provide hints
After three incorrect guesses, the game gives a divisibility hint. The hint is based on the secret number, but it does not reveal the number itself.


In [ ]:
def divisibility_hint(secret_number):
    """Return a useful divisibility hint for the secret number."""
    # Check larger divisors first so the hint gives the player useful information.
    for divisor in (10, 9, 8, 7, 6, 5, 4, 3, 2):
        # A remainder of zero means the secret number divides evenly by this value.
        if secret_number % divisor == 0:
            return f"Hint: The number is divisible by {divisor}."

    # This fallback applies when the number has no divisor from 2 through 10.
    return "Hint: The number is not divisible by any number from 2 through 10."



`random.randint(1, 100)` creates the secret number, including both endpoints.

A valid guess increases the attempt count. Invalid input, such as text or a number outside 1 through 100, does not use a guess.

The game ends immediately when the player guesses correctly or when the selected number of guesses has been used.


In [ ]:
# Import random so each game receives a new secret number.
import random


def play_guessing_game():
    """Run one complete game and return the player's score information."""
    # Introduce the game and explain the range of possible numbers.
    print("\nGuess the Number")
    print("I am thinking of a number from 1 to 100.")

    # Choose the difficulty and use it to determine the number of guesses.
    difficulty = choose_difficulty()
    max_attempts = DIFFICULTY_LIMITS[difficulty]

    # Generate the hidden target and reset counters for this round.
    secret_number = random.randint(1, 100)
    attempts = 0
    wrong_guesses = 0

    print(f"You have {max_attempts} guesses on {difficulty.title()} mode.")

    # Continue until the player wins or uses every allowed valid guess.
    while attempts < max_attempts:
        try:
            # Convert the player's text input into an integer guess.
            guess = int(input("Enter your guess: "))
        except ValueError:
            # Text that is not a whole number should not consume an attempt.
            print("Please enter a whole number.")
            continue

        # Reject numbers outside the game's promised 1-to-100 range.
        if not 1 <= guess <= 100:
            print("Your guess must be between 1 and 100.")
            continue

        # Count only valid guesses and calculate what remains afterward.
        attempts += 1
        remaining_guesses = max_attempts - attempts

        if guess == secret_number:
            # Reward faster correct guesses with a higher score.
            score = max_attempts - attempts + 1
            print(f"Correct! You won in {attempts} attempt(s).")
            print(f"Score: {score}")
            return {
                "won": True,
                "attempts": attempts,
                "score": score,
                "difficulty": difficulty,
            }

        # Record an incorrect guess and tell the player which direction to try.
        wrong_guesses += 1
        if guess > secret_number:
            print("Too High")
        else:
            print("Too Low")

        # Give the optional divisibility hint after exactly three wrong guesses.
        if wrong_guesses == 3:
            print(divisibility_hint(secret_number))

        # Show the remaining number of valid guesses unless the game is over.
        if remaining_guesses:
            print(f"Remaining guesses: {remaining_guesses}")

    # Return a consistent result when the player uses all attempts without winning.
    print(f"Game over! The number was {secret_number}.")
    return {
        "won": False,
        "attempts": attempts,
        "score": 0,
        "difficulty": difficulty,
    }


## 4. Save high scores

A winning score stores the username, score, difficulty, and number of attempts in `guess_number_scores.txt`.

The file uses one score per line so it can be opened and read without any extra library.


In [ ]:
def save_high_score(username, game_result):
    """Append a winning result to the score file."""
    # Do not save games that ended without a correct guess.
    if not game_result["won"]:
        return

    # Remove separators and line breaks so each score stays on one file line.
    clean_username = username.strip().replace("|", " ").replace("\n", " ")

    # Store the fields in a predictable format for the leaderboard to read later.
    score_line = (
        f"{clean_username}|{game_result['score']}|"
        f"{game_result['difficulty']}|{game_result['attempts']}\n"
    )

    # Append the new score without deleting scores from earlier games.
    with SCORES_FILE.open("a", encoding="utf-8") as score_file:
        score_file.write(score_line)

    print(f"High score saved in {SCORES_FILE}.")


## 5. Build a leaderboard

The leaderboard reads saved scores, ignores malformed lines, sorts the valid scores from highest to lowest, and displays the top ten players.


In [ ]:
def load_scores():
    """Read valid scores from the text file and return them as dictionaries."""
    # Start with an empty list when no score file has been created yet.
    scores = []
    if not SCORES_FILE.exists():
        return scores

    # Read each saved line and convert it back into structured score data.
    with SCORES_FILE.open("r", encoding="utf-8") as score_file:
        for line in score_file:
            # Every valid line contains username, score, difficulty, and attempts.
            parts = line.strip().split("|")
            if len(parts) != 4:
                # Skip incomplete or incorrectly formatted lines.
                continue

            username, score, difficulty, attempts = parts
            try:
                # Convert numeric text into integers for sorting and display.
                scores.append(
                    {
                        "username": username,
                        "score": int(score),
                        "difficulty": difficulty,
                        "attempts": int(attempts),
                    }
                )
            except ValueError:
                # Ignore lines whose score or attempt values are not numbers.
                continue

    return scores


def display_leaderboard():
    """Print the ten highest scores."""
    # Sort the saved results so the highest score appears first.
    scores = sorted(
        load_scores(),
        key=lambda result: result["score"],
        reverse=True,
    )

    print("\nLeaderboard")
    if not scores:
        # Explain why there is no ranking when no valid scores exist.
        print("No scores saved yet.")
        return

    # Display only the ten best results, with positions starting at one.
    for position, result in enumerate(scores[:10], start=1):
        print(
            f"{position}. {result['username']} - "
            f"{result['score']} points ({result['difficulty'].title()}, "
            f"{result['attempts']} attempts)"
        )


## 6. Run a game and update the leaderboard

Run the final cell to start playing. After a win, enter a username to save the score. The leaderboard is shown after every round.


In [ ]:
def run_game_session():
    """Play repeated rounds until the player chooses to stop."""
    # Keep starting rounds until the player enters something other than y.
    while True:
        # Play one round and receive its win, score, difficulty, and attempt data.
        result = play_guessing_game()

        if result["won"]:
            # Ask for a name only after a successful game.
            username = input("Enter your username for the leaderboard: ").strip()
            if username:
                # Save the completed result so it appears in future leaderboards.
                save_high_score(username, result)
            else:
                # Avoid creating an anonymous leaderboard entry.
                print("No username entered; score was not saved.")

        # Show the current rankings after every completed round.
        display_leaderboard()
        play_again = input("Play again? (y/n): ").strip().lower()
        if play_again != "y":
            # End the session when the player does not choose another round.
            print("Thanks for playing!")
            break


# Start the interactive game when this final cell is run.
run_game_session()



Guess the Number
I am thinking of a number from 1 to 100.
You have 10 guesses on Easy mode.
Too Low
Remaining guesses: 9
Too High
Remaining guesses: 8
Too Low
Hint: The number is divisible by 3.
Remaining guesses: 7
Too Low
Remaining guesses: 6
Too Low
Remaining guesses: 5
Too Low
Remaining guesses: 4
Correct! You won in 7 attempt(s).
Score: 4
High score saved in guess_number_scores.txt.

Leaderboard
1. Mwariri - 4 points (Easy, 7 attempts)
Thanks for playing!
